In [34]:
import json
import numpy as np
import pandas as pd
import pickle5 as pickle
import copy

In [35]:
#pip show pickle5

In [36]:
rdf_all = pd.read_csv('./modcloth/df_modcloth.csv', sep=",")
rdf_all['user_id'] = rdf_all['user_id'].astype('category').cat.codes
rdf_all.head(100)

,item_id,user_id,rating,timestamp,size,fit,user_attr,model_attr,category,brand,year,split
0,7443,309,4,2010-01-21 08:00:00+00:00,NaN,NaN,Small,Small,Dresses,NaN,2012,0
1,7443,13009,3,2010-01-27 08:00:00+00:00,NaN,NaN,NaN,Small,Dresses,NaN,2012,0
2,7443,5534,4,2010-01-29 08:00:00+00:00,NaN,NaN,Small,Small,Dresses,NaN,2012,0
3,7443,1716,4,2010-02-13 08:00:00+00:00,NaN,NaN,NaN,Small,Dresses,NaN,2012,0
4,7443,42071,4,2010-02-18 08:00:00+00:00,NaN,NaN,Small,Small,Dresses,NaN,2012,0
...,...,...,...,...,...,...,...,...,...,...,...,...
95,7443,384,4,2010-11-06 07:00:00+00:00,NaN,Slightly small,Small,Small,Dresses,NaN,2012,0
96,7443,10111,3,2010-11-06 07:00:00+00:00,NaN,Slightly small,NaN,Small,Dresses,NaN,2012,0
97,7443,26577,4,2010-11-06 07:00:00+00:00,NaN,Just right,Large,Small,Dresses,NaN,2012,0
98,11960,42721,3,2010-11-06 07:00:00+00:00,NaN,Just right,NaN,Small&Large,Outerwear,NaN,2010,2


In [37]:
rdf_all = rdf_all.dropna(subset=['user_attr'])
rdf_all.reset_index(drop=True, inplace=True)
rdf_all.head()

,item_id,user_id,rating,timestamp,size,fit,user_attr,model_attr,category,brand,year,split
0,7443,309,4,2010-01-21 08:00:00+00:00,NaN,NaN,Small,Small,Dresses,NaN,2012,0
1,7443,5534,4,2010-01-29 08:00:00+00:00,NaN,NaN,Small,Small,Dresses,NaN,2012,0
2,7443,42071,4,2010-02-18 08:00:00+00:00,NaN,NaN,Small,Small,Dresses,NaN,2012,0
3,7443,3485,2,2010-02-26 08:00:00+00:00,NaN,NaN,Small,Small,Dresses,NaN,2012,0
4,7443,1964,4,2010-04-06 07:00:00+00:00,NaN,NaN,Small,Small,Dresses,NaN,2012,0


In [38]:
rdf = rdf_all[["user_id","item_id", "rating"]]

rdf.head()


,user_id,item_id,rating
0,309,7443,4
1,5534,7443,4
2,42071,7443,4
3,3485,7443,2
4,1964,7443,4


In [39]:
user_df = rdf_all[["user_id", "user_attr"]]

user_df.shape

(91526, 2)

In [40]:
user_df.head()


,user_id,user_attr
0,309,Small
1,5534,Small
2,42071,Small
3,3485,Small
4,1964,Small


In [41]:
user_type_dict = dict()
for u in range(len(user_df)):
    type_str = user_df.at[u, "user_attr"]
    
    #type_list = type_str.split('|')
    type_list = type_str
    user_type_dict[user_df.at[u, 'user_id']] = [type_list]

In [42]:
print(user_type_dict)

{309: ['Small'], 5534: ['Small'], 42071: ['Small'], 3485: ['Small'], 1964: ['Small'], 6862: ['Large'], 380: ['Small'], 10927: ['Small'], 27219: ['Small'], 36776: ['Small'], 3897: ['Small'], 4774: ['Small'], 6488: ['Small'], 4617: ['Small'], 6330: ['Small'], 13885: ['Small'], 4985: ['Small'], 2759: ['Small'], 1672: ['Small'], 2058: ['Small'], 6692: ['Small'], 857: ['Small'], 3415: ['Small'], 3520: ['Small'], 5891: ['Small'], 1250: ['Small'], 3089: ['Small'], 5022: ['Small'], 1176: ['Small'], 5129: ['Small'], 2465: ['Small'], 5924: ['Small'], 6316: ['Small'], 339: ['Small'], 5568: ['Small'], 5168: ['Small'], 124: ['Small'], 6509: ['Small'], 6546: ['Small'], 697: ['Small'], 852: ['Small'], 3578: ['Small'], 6476: ['Small'], 41159: ['Small'], 6185: ['Small'], 1479: ['Small'], 6720: ['Small'], 1958: ['Large'], 246: ['Large'], 384: ['Small'], 26577: ['Large'], 38400: ['Small'], 928: ['Small'], 8073: ['Small'], 320: ['Small'], 4770: ['Small'], 4171: ['Small'], 1222: ['Small'], 5456: ['Small'],

In [43]:
item_set = set(rdf['item_id'].unique())
user_set = set(rdf['user_id'].unique())
print('item num = ' + str(len(item_set)))
print('user num = ' + str(len(user_set)))

item num = 1019
user num = 39536


In [44]:
# count the number for each user type and sort
import operator
type_count = dict()
for l in user_type_dict:
    for g in user_type_dict[l]:
        if not g in type_count:
            type_count[g] = 1
        else:
            type_count[g] += 1

type_count_sorted = sorted(type_count.items(), key=operator.itemgetter(1), reverse=True)
type_count_sorted

[('Small', 30141), ('Large', 9395)]

In [45]:
key_type = ['Small', 'Large']

# get the key_type->user_list dict
key_type_user = dict()
for k in key_type:
    key_type_user[k] = list()
for user in user_type_dict:
    for t in user_type_dict[user]:
        if t in key_type:
            key_type_user[t].append(user)

In [46]:
# collect all the users with key types
key_user_set = set()
for types in key_type_user:
    key_user_set |= set(key_type_user[types])

nonkey_user_set = user_set - key_user_set

In [47]:
# remove the non-key type users in rdf
remove_list = []
for user in nonkey_user_set:
    remove_list += rdf.index[rdf['user_id'] == user].values.tolist()

In [48]:
rdf.drop(remove_list, inplace=True)

C:\ProgramData\Anaconda3\lib\site-packages\pandas\core\frame.py:3997: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  errors=errors,


In [49]:
rdf.reset_index(drop=True, inplace=True)
rating_df = copy.copy(rdf)

In [50]:
rdf = copy.copy(rating_df)

In [51]:
# iteratively remove items and users with less than 2 reviews
rdf.reset_index(drop=True, inplace=True)

rdf['user_freq'] = rdf.groupby('user_id')['user_id'].transform('count')
rdf.drop(rdf.index[rdf['user_freq'] <= 2], inplace=True)
rdf.reset_index(drop=True, inplace=True)
rdf['item_freq'] = rdf.groupby('item_id')['item_id'].transform('count')
rdf.drop(rdf.index[rdf['item_freq'] <= 2], inplace=True)
rdf.reset_index(drop=True, inplace=True)
rdf['user_freq'] = rdf.groupby('user_id')['user_id'].transform('count')
rdf.reset_index(drop=True, inplace=True)
rdf['user_id'].value_counts()

5756     248
411      203
22660    198
3735     195
2907     191
        ... 
7823       2
18032      2
23406      2
32011      2
4092       2
Name: user_id, Length: 6339, dtype: int64

In [52]:
item_list = rdf['item_id'].unique()
user_list = rdf['user_id'].unique()
print('item num = ' + str(len(item_list)))
print('user num = ' + str(len(user_list)))

item num = 915
user num = 6339


In [53]:
# get the user and item str id->int id dict
i = 0
user_id_dict = dict()
for u in user_list:
    if not u in user_id_dict:
        user_id_dict[u] = i
        i += 1
j = 0
item_id_dict = dict()
for i in item_list:
    if not i in item_id_dict:
        item_id_dict[i] = j
        j += 1

In [54]:
print('sparsity: ' + str(len(rdf) * 1.0 / (len(user_list) * len(item_list))))

sparsity: 0.008835580244423238


In [55]:
# get the df of train, vali, and test set
rdf.reset_index(inplace=True, drop=True)
train_df = rdf.copy()
vali_df = rdf.copy()
test_df = rdf.copy()

train_ratio = 0.8
vali_ratio = 0.0
test_ratio = 0.2
num_all = len(rdf)
vali_idx = []
test_idx = []

test_vali_idx = []
i = 0
num_user = len(user_list)
for u in user_list:
    u_idx = train_df.index[train_df['user_id'] == u]
    idx_len = len(u_idx)
    test_len = int(idx_len * (test_ratio + vali_ratio))
    if test_len == 0:
        test_len = 1
    tmp = np.random.choice(u_idx, size=test_len, replace=False)
    test_vali_idx += tmp.tolist()
    i += 1
    if i % 5000 == 0:
        print(str(i) + '/' + str(num_user))

# tmp = (np.random.choice(range(num_all), size=(test_len+vali_len), replace=False)).tolist()
test_len = int(len(test_vali_idx) * test_ratio / (test_ratio + vali_ratio))
vali_len = int(len(test_vali_idx) - test_len)
test_idx = (np.random.choice(test_vali_idx, size=test_len, replace=False)).tolist()
vali_idx = (np.random.choice(test_vali_idx, size=vali_len, replace=False)).tolist()

test_set = set(test_idx)
vali_set = set(vali_idx)
train_set = set(range(num_all)) - test_set - vali_set
train_idx = list(train_set)
train_df.drop((test_idx + vali_idx), axis=0, inplace=True)
test_df.drop((train_idx + vali_idx), axis=0, inplace=True)
vali_df.drop((train_idx + test_idx), axis=0, inplace=True)

5000/6339


In [56]:
train_df.shape

(40354, 5)

In [76]:
len(train_df['user_id'].unique())

6339

In [57]:
# get the matrix of train, vali and test set

train_df.reset_index(drop=True, inplace=True)
test_df.reset_index(drop=True, inplace=True)
vali_df.reset_index(drop=True, inplace=True)
rdf.reset_index(drop=True, inplace=True)
train = np.zeros((len(user_list), len(item_list)))
test = np.zeros((len(user_list), len(item_list)))
vali = np.zeros((len(user_list), len(item_list)))
for r in range(len(train_df)):
    train[user_id_dict[train_df.at[r, 'user_id']], item_id_dict[train_df.at[r, 'item_id']]] = 1.0
for r in range(len(test_df)):
    test[user_id_dict[test_df.at[r, 'user_id']], item_id_dict[test_df.at[r, 'item_id']]] = 1.0
for r in range(len(vali_df)):
    vali[user_id_dict[vali_df.at[r, 'user_id']], item_id_dict[vali_df.at[r, 'item_id']]] = 1.0

In [58]:
test

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [59]:
# get the user int id-> str id list, and the same for item 
user_list = user_id_dict.keys()
user_idd_list = list()
for u in range(len(user_list)):
    user_idd_list.append('')
for user in user_id_dict:
    user_idd_list[user_id_dict[user]] = user

item_list = item_id_dict.keys()
item_idd_list = list()
for i in range(len(item_list)):
    item_idd_list.append('')
for item in item_id_dict:
    item_idd_list[item_id_dict[item]] = item
    
# get the user int id->types list
user_idd_type_list = list()
for u in range(len(user_idd_list)):
    user_idd_type_list.append(user_type_dict[user_idd_list[u]])

In [60]:
train_df.head()

,user_id,item_id,rating,user_freq,item_freq
0,309,7443,4,66,431
1,5534,7443,4,30,431
2,42071,7443,4,12,431
3,3485,7443,2,71,431
4,1964,7443,4,11,431


In [61]:
train_df.drop('user_freq', axis=1, inplace=True)
train_df.drop('item_freq', axis=1, inplace=True)
vali_df.drop('user_freq', axis=1, inplace=True)
vali_df.drop('item_freq', axis=1, inplace=True)
test_df.drop('user_freq', axis=1, inplace=True)
test_df.drop('item_freq', axis=1, inplace=True)
rdf.drop('user_freq', axis=1, inplace=True)
rdf.drop('item_freq', axis=1, inplace=True)

In [62]:
train_df.head()

,user_id,item_id,rating
0,309,7443,4
1,5534,7443,4
2,42071,7443,4
3,3485,7443,2
4,1964,7443,4


In [63]:
# get df for rdf, train, vali, test with int id for user and item
import copy
rating_df = copy.copy(rdf)
for i in range(len(rdf)):
    rating_df.at[i, 'user_id'] = user_id_dict[rating_df.at[i, 'user_id']]
    rating_df.at[i, 'item_id'] = item_id_dict[rating_df.at[i, 'item_id']]

training_df = copy.copy(train_df)
for i in range(len(training_df)):
    training_df.at[i, 'user_id'] = user_id_dict[training_df.at[i, 'user_id']]
    training_df.at[i, 'item_id'] = item_id_dict[training_df.at[i, 'item_id']]

valiing_df = copy.copy(vali_df)
for i in range(len(valiing_df)):
    valiing_df.at[i, 'user_id'] = user_id_dict[valiing_df.at[i, 'user_id']]
    valiing_df.at[i, 'item_id'] = item_id_dict[valiing_df.at[i, 'item_id']]

testing_df = copy.copy(test_df)
for i in range(len(testing_df)):
    testing_df.at[i, 'user_id'] = user_id_dict[testing_df.at[i, 'user_id']]
    testing_df.at[i, 'item_id'] = item_id_dict[testing_df.at[i, 'item_id']]

In [64]:
# generate the rating list for each key type, get the type->ratings dict
rdf.reset_index(drop=True, inplace=True)
key_type_rating = dict()
for k in key_type:
    key_type_rating[k] = 0.0
for r in range(len(rdf)):
    user = rdf.at[r, 'user_id']
    gl = user_type_dict[user]
    for k in key_type:
        if k in gl:
            key_type_rating[k] += 1.0

# get the user int id->type list
type_user_vector = dict()
for k in key_type:
    type_user_vector[k] = np.zeros((1, len(user_list)))
for u in range(len(user_idd_type_list)):
    type_list = user_idd_type_list[u]
    for t in type_list:
        if t in key_type:
            type_user_vector[t][0,u] = 1.0

In [65]:
with open("user_type_dict_modcloth.pkl", "wb") as f:
    pickle.dump(user_type_dict, f)
with open("type_user_vector_modcloth.pkl", "wb") as f:
    pickle.dump(type_user_vector, f, pickle.HIGHEST_PROTOCOL)
with open("key_type_modcloth.pkl", "wb") as f:
    pickle.dump(key_type, f, pickle.HIGHEST_PROTOCOL)
with open("user_id_dict_modcloth.pkl", "wb") as f:
    pickle.dump(user_id_dict, f, pickle.HIGHEST_PROTOCOL)
with open("item_id_dict_modcloth.pkl", "wb") as f:
    pickle.dump(item_id_dict, f, pickle.HIGHEST_PROTOCOL)
# with open("rdf.pkl", "wb") as f:
#     pickle.dump(rdf, f, pickle.HIGHEST_PROTOCOL)
with open("rating_df_modcloth.pkl", "wb") as f:
    pickle.dump(rating_df, f, pickle.HIGHEST_PROTOCOL)
with open("training_df_modcloth.pkl", "wb") as f:
    pickle.dump(training_df, f, pickle.HIGHEST_PROTOCOL)
with open("valiing_df_modcloth.pkl", "wb") as f:
    pickle.dump(valiing_df, f, pickle.HIGHEST_PROTOCOL)
with open("testing_df_modcloth.pkl", "wb") as f:
    pickle.dump(testing_df, f, pickle.HIGHEST_PROTOCOL)
with open("user_idd_type_list_modcloth.pkl", "wb") as f:
    pickle.dump(user_idd_type_list, f, pickle.HIGHEST_PROTOCOL)
with open("item_idd_list_modcloth.pkl", "wb") as f:
    pickle.dump(item_idd_list, f, pickle.HIGHEST_PROTOCOL)
with open("user_idd_list_modcloth.pkl", "wb") as f:
    pickle.dump(user_idd_list, f, pickle.HIGHEST_PROTOCOL)
with open("key_type_rating_modcloth.pkl", "wb") as f:
    pickle.dump(key_type_rating, f, pickle.HIGHEST_PROTOCOL)
    
with open("train_modcloth.mat", "wb") as f:
    np.save(f, train)
with open("test_modcloth.mat", "wb") as f:
    np.save(f, test)
with open("vali_modcloth.mat", "wb") as f:
    np.save(f, vali)

In [68]:
# count the number for each user type and sort
import pickle5 as pickle
from operator import itemgetter
user_list = rdf['user_id'].unique()
#user_type_dict = pickle.load(open('./user_type_dict.pkl'))

with open('./user_type_dict_modcloth.pkl', 'rb') as f:
    user_type_dict = pickle.load(f,encoding='latin1')

with open('./key_type_modcloth.pkl', 'rb') as f:
    key_type = pickle.load(f,encoding='latin1')

#key_type = pickle.load(open('./key_type.pkl'))

type_count = dict()
for u in user_list:
    gl = user_type_dict[u]
    for g in gl:
        if g in key_type:
            if not g in type_count:
                type_count[g] = 1
            else:
                type_count[g] += 1

# with open("genre_count.pkl", "wb") as f:
#     pickle.dump(genre_count, f, pickle.HIGHEST_PROTOCOL)
                
type_count_sorted = sorted(type_count.items(), key=itemgetter(1), reverse=True)
type_count_sorted

[('Small', 4614), ('Large', 1725)]

In [69]:
with open("type_count_modcloth.pkl", "wb") as f:
    pickle.dump(type_count_sorted, f)

In [70]:
import numpy as np
import pickle5 as pickle
import copy as copy
train = np.load('./train_modcloth.mat')

with open('./user_idd_type_list_modcloth.pkl', 'rb') as f:
    user_idd_type_list = pickle.load(f,encoding='latin1')

user_idd_type_list = np.array(user_idd_type_list)


mask = 1.0 * (train > 0)
item_type_count = list()
for i in range(train.shape[1]):
    temp_type_count = copy.copy(type_count)
    mask_i = mask[:, i]
    gll = user_idd_type_list[mask_i == 1.0]
    for gl in gll:
        for g in gl:
            if g in key_type:
                temp_type_count[g] -= 1
    item_type_count.append(temp_type_count)
# with open("user_genre_count.pkl", "wb") as f:
#     pickle.dump(user_genre_count, f, pickle.HIGHEST_PROTOCOL)

In [71]:
with open("item_type_count_modcloth.pkl", "wb") as f:
    pickle.dump(item_type_count, f)

In [72]:
len(item_type_count)

915

In [73]:
type_count_sorted

[('Small', 4614), ('Large', 1725)]

In [74]:
with open('./key_type_rating_modcloth.pkl', 'rb') as f:
    key_type_rating = pickle.load(f,encoding='latin1')


#key_type_rating = pickle.load(open('./key_genre_rating.pkl'))
type_avg_like = dict()
for k in key_type:
    type_avg_like[k] = key_type_rating[k] * 1.0 / type_count[k]

In [75]:
type_avg_like_sorted = sorted(type_avg_like.items(), key=itemgetter(1), reverse=True)
type_avg_like_sorted

[('Small', 9.178803641092328), ('Large', 5.15768115942029)]